# Multimodal Financial Runway Assistant

An interactive financial assistant that combines structured tool calling, deterministic Python calculations, and generated speech.

The assistant gathers a user's income, expenses, savings, investments, debts, and financial goal. Once the required information is available, the language model calls `calculate_runway()` instead of estimating the result itself. The final answer is presented as both text and spoken audio in a Gradio interface.

> **Note:** This application provides educational estimates, not personalized financial advice.

## 1. Dependencies

The application uses the OpenAI Python client for language and speech generation, JSON for tool arguments, dotenv for local configuration, and Gradio for the user interface.

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

## 2. Configuration

The OpenAI API key is loaded from a local `.env` file and is never displayed in the notebook output.

In [ ]:
# Load environment variables (OPENAI_API_KEY should be in your .env file)
load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
if not openai_api_key:
    raise RuntimeError("OPENAI_API_KEY is not set. Add it to your .env file.")
print("OpenAI API key configured.")

MODEL = "gpt-4.1-mini"
openai = OpenAI(api_key=openai_api_key)

# Optional local-model configuration:
# MODEL = "llama3.2"
# openai = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

In [ ]:
# System prompt for the Financial Runway Assistant
# (Carried over from the original project — the LLM gathers info before calling the tool)

system_message = """
You are a Financial Runway Assistant.

Gather information about the potential client's:
- income
- expenses
- cash savings
- investments
- debts
- financial goals

Ask no more than two questions at a time.
Do not calculate financial runway until the required information is available.
When you have enough data, call the calculate_runway tool.
Always be accurate. If you don't know the answer, say so.
"""

## 3. The `calculate_runway()` function

Financial runway is calculated as accessible assets divided by monthly expenses. Accessible assets include cash savings and investments that could be used if employment income stopped. The calculation is performed in Python so the numeric result is deterministic rather than generated by the language model.

In [ ]:
def calculate_runway(
    monthly_income: float,
    monthly_expenses: float,
    cash_savings: float,
    investments: float,
    debts: float,
    financial_goal: str,
) -> str:
    """
    Calculate how long accessible assets could cover monthly expenses if income stopped today.

    Income, debt, and the stated goal give the assistant conversational context;
    the runway formula uses accessible assets and monthly expenses.

    Args:
        monthly_income:     Net monthly income in dollars
        monthly_expenses:   Average monthly expenses in dollars
        cash_savings:       Total cash savings in dollars
        investments:        Total accessible investment assets in dollars
        debts:              Total outstanding debt in dollars
        financial_goal:     The client's stated financial goal

    Returns:
        A human-readable string describing the runway estimate and assumptions.
    """
    print(
        f"Tool called: calculate_runway(expenses={monthly_expenses}, "
        f"savings={cash_savings}, investments={investments})"
    )

    # Basic validation — cannot divide by zero or negative expenses
    if monthly_expenses <= 0:
        return (
            "Cannot calculate runway: monthly expenses must be greater than zero. "
            "Please provide a valid monthly expense amount."
        )

    # Accessible assets = cash on hand plus investments you could tap if needed
    total_accessible_assets = cash_savings + investments

    # Runway = how many months (and years) those assets would last at current spending
    runway_months = total_accessible_assets / monthly_expenses
    runway_years = runway_months / 12

    return (
        f"If employment income stopped today, your accessible assets "
        f"(${total_accessible_assets:,.0f} in cash and investments) could cover "
        f"your current monthly expenses (${monthly_expenses:,.0f}) for approximately "
        f"{runway_months:.1f} months ({runway_years:.1f} years)."
    )

## 4. Tool definition for the LLM

The schema exposes `calculate_runway()` to the model and requires all six financial fields. `additionalProperties` is disabled so tool requests remain predictable and match the Python function signature.

In [ ]:
# Schema describing calculate_runway for the OpenAI tools API

runway_function = {
    "name": "calculate_runway",
    "description": "Calculate the client's financial runway once all required financial data has been gathered.",
    "parameters": {
        "type": "object",
        "properties": {
            "monthly_income": {
                "type": "number",
                "description": "Net monthly income in dollars",
            },
            "monthly_expenses": {
                "type": "number",
                "description": "Average monthly expenses in dollars",
            },
            "cash_savings": {
                "type": "number",
                "description": "Total cash savings in dollars",
            },
            "investments": {
                "type": "number",
                "description": "Total accessible investment assets in dollars",
            },
            "debts": {
                "type": "number",
                "description": "Total outstanding debt in dollars",
            },
            "financial_goal": {
                "type": "string",
                "description": "The client's stated financial goal",
            },
        },
        "required": [
            "monthly_income",
            "monthly_expenses",
            "cash_savings",
            "investments",
            "debts",
            "financial_goal",
        ],
        "additionalProperties": False,
    },
}

# Wrap the function schema in the tools list passed to chat.completions.create()
tools = [{"type": "function", "function": runway_function}]

## 5. Tool execution and conversation loop

`handle_tool_calls()` converts streamed JSON arguments into Python values, executes `calculate_runway()`, and links the result to the original request. `chat()` preserves the conversation, reconstructs any tool calls received across multiple chunks, and yields the growing final answer so it appears as the model generates it.

In [ ]:
def handle_tool_calls(tool_calls):
    """
    Execute tool calls requested by the LLM and return tool-result messages.
    """
    responses = []

    for tool_call in tool_calls:
        function = tool_call["function"]
        if function["name"] == "calculate_runway":
            # The API supplies arguments as a JSON string, so convert it to a Python dict.
            arguments = json.loads(function["arguments"])

            # ** unpacks the dictionary into named arguments for calculate_runway().
            result = calculate_runway(**arguments)

            # The tool_call_id links this result to the LLM's original request.
            responses.append({
                "role": "tool",
                "content": result,
                "tool_call_id": tool_call["id"],
            })

    return responses

In [ ]:
def chat(message, history):
    """Run the conversation and tools, yielding the final text as it streams."""
    history = history or []
    messages = [{"role": "system", "content": system_message}]
    messages.extend(
        {"role": item["role"], "content": item["content"]}
        for item in history
    )
    messages.append({"role": "user", "content": message})

    tool_rounds = 0
    while True:
        stream = openai.chat.completions.create(
            model=MODEL, messages=messages, tools=tools, stream=True
        )

        full_response = ""
        tool_call_buffers = {}

        for chunk in stream:
            delta = chunk.choices[0].delta

            if delta.content:
                full_response += delta.content
                yield full_response

            # Tool-call names and JSON arguments may arrive across several chunks.
            for tool_call in delta.tool_calls or []:
                buffered = tool_call_buffers.setdefault(
                    tool_call.index,
                    {"id": "", "type": "function", "function": {"name": "", "arguments": ""}},
                )
                if tool_call.id:
                    buffered["id"] = tool_call.id
                if tool_call.function and tool_call.function.name:
                    buffered["function"]["name"] += tool_call.function.name
                if tool_call.function and tool_call.function.arguments:
                    buffered["function"]["arguments"] += tool_call.function.arguments

        if not tool_call_buffers:
            if not full_response:
                yield "I could not generate a response."
            return

        tool_rounds += 1
        if tool_rounds > 5:
            yield "I could not complete the calculation because too many tool calls were requested."
            return

        tool_calls = [tool_call_buffers[index] for index in sorted(tool_call_buffers)]
        messages.append({
            "role": "assistant",
            "content": full_response or None,
            "tool_calls": tool_calls,
        })
        messages.extend(handle_tool_calls(tool_calls))

## 6. Text-to-speech

The final assistant response is converted to speech with OpenAI's `gpt-4o-mini-tts` model. The function returns MP3 audio bytes that can be passed directly to the Gradio audio component.

In [ ]:
def talker(message):
    """Convert the assistant's final text response into MP3 audio bytes."""
    speech = openai.audio.speech.create(
        model="gpt-4o-mini-tts",
        voice="onyx",
        input=message,
    )
    return speech.content

## 7. Multimodal Gradio interface

The Gradio interface maintains message history and updates the assistant message as streamed text arrives. After the response is complete, it converts the final text to speech and automatically plays the generated audio.

In [ ]:
def respond(message, history):
    """Stream the visible reply, then produce audio from the completed text."""
    history = history or []
    conversation = history + [{"role": "user", "content": message}]
    reply = ""

    for reply in chat(message, history):
        updated_history = conversation + [
            {"role": "assistant", "content": reply}
        ]
        yield "", updated_history, None

    # Generate speech only once, after the streamed response is complete.
    yield "", updated_history, talker(reply)


with gr.Blocks() as demo:
    gr.Markdown("# Multimodal Financial Runway Assistant")
    gr.Markdown(
        "Share your income, expenses, savings, investments, debts, and goal "
        "to receive a tool-calculated runway estimate in text and audio."
    )
    chatbot = gr.Chatbot(type="messages", height=500)
    audio = gr.Audio(label="Spoken response", autoplay=True)
    message_box = gr.Textbox(
        label="Your message",
        placeholder="Tell me about your financial situation.",
    )
    message_box.submit(
        respond,
        inputs=[message_box, chatbot],
        outputs=[message_box, chatbot, audio],
    )

# Run this line after executing the notebook cells:
demo.launch()